In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Synthetic datasets
df_tx = pd.DataFrame({
    'customer_id': [f'C-{i:04d}' for i in range(1, 4821)],
    'churned': np.random.randint(0, 2, 4820),
})

# Profiles table: 4800 unique customers + 3 duplicates for C-0099
profile_ids = [f'C-{i:04d}' for i in range(1, 4801)] + ['C-0099', 'C-0099']  # C-0099 appears 3 times
df_profiles = pd.DataFrame({
    'customer_id': profile_ids,
    'company_size': np.random.choice(['SMB', 'Enterprise'], len(profile_ids)),
})

print(f"df_tx rows: {len(df_tx)}")
print(f"df_profiles rows: {len(df_profiles)}")

## Bug 1: Inner join silently drops non-matching rows

In [ ]:
# --- BUGGY CODE ---
df_merged_inner = pd.merge(df_tx, df_profiles, on='customer_id', how='inner')
print(f"Rows after inner join: {len(df_merged_inner)}")
print(f"Rows dropped: {len(df_tx) - len(df_merged_inner)}")

**Detect:** The 37 unmatched rows are silently dropped. Which customers are they?

In [ ]:
# Detect missing customers
unmatched = df_tx[~df_tx['customer_id'].isin(df_merged_inner['customer_id'])]
print(f"Unmatched customer IDs: {len(unmatched)}")
print(f"Explanation: Write here why inner join is wrong for this use case.")

**Fix:**

In [ ]:
# Correct: use left join to keep all transaction rows
df_merged = pd.merge(df_tx, df_profiles, on='customer_id', how='left')
print(f"Rows after left join: {len(df_merged)}")
print(f"Missing profiles (NaN): {df_merged['company_size'].isnull().sum()}")

## Bug 2: Missing `validate=` causes silent row explosion

In [ ]:
# --- BUGGY CODE ---
df_merged_buggy = pd.merge(df_tx, df_profiles, on='customer_id', how='left')
print(f"Merged rows: {len(df_merged_buggy)}")
print(f"Expected: 4820, Got: {len(df_merged_buggy)}")
print(f"Extra rows: {len(df_merged_buggy) - len(df_tx)}")

**Detect:** C-0099 appears multiple times in profiles:

In [ ]:
# Find the row explosion
c0099_rows = df_merged_buggy[df_merged_buggy['customer_id'] == 'C-0099']
print(f"C-0099 appears {len(c0099_rows)} times in merged data")
print(f"Explanation: Write here why this is a silent data corruption bug.")

**Fix:**

In [ ]:
# Use validate to catch the duplicate
try:
    df_merged_validated = pd.merge(df_tx, df_profiles, on='customer_id', how='left', validate='one_to_one')
except pd.errors.MergeError as e:
    print(f"MergeError (expected): {e}")
    print(f"\nFix: Resolve the duplicate in df_profiles first")
    df_profiles_clean = df_profiles.drop_duplicates(subset=['customer_id'], keep='first')
    df_merged_fixed = pd.merge(df_tx, df_profiles_clean, on='customer_id', how='left', validate='one_to_one')
    print(f"Merged rows after fix: {len(df_merged_fixed)}")

## Bug 3: Merging before aggregating long-format data

In [ ]:
# Create monthly spend data
spend_data = []
for cid in [f'C-{i:04d}' for i in range(1, 101)]:
    for month in range(1, 13):
        spend_data.append({'customer_id': cid, 'month': month, 'spend': np.random.uniform(50, 200)})

df_long = pd.DataFrame(spend_data)

print(f"Long-format spend data (12 months × 100 customers): {len(df_long)} rows")

In [ ]:
# --- BUGGY CODE ---
# Merge BEFORE aggregating
df_merged_long = pd.merge(df_tx[:100], df_long, on='customer_id', how='left')
print(f"Merged rows: {len(df_merged_long)}")
print(f"Each transaction now has 12 rows (one per month)")
print(f"Explanation: Write here why this is a data multiplication bug.")

**Fix:**

In [ ]:
# Aggregate BEFORE merging
trailing_spend = (df_long.groupby('customer_id')['spend']
                  .apply(lambda s: s.tail(3).mean())
                  .rename('trailing_3m_spend')
                  .reset_index())

print(f"Aggregated to one row per customer: {len(trailing_spend)} rows")

df_merged_correct = pd.merge(df_tx[:100], trailing_spend, on='customer_id', how='left')
print(f"Final merged rows: {len(df_merged_correct)}")
print(f"Each transaction has exactly one row")

## Summary check

In [ ]:
print("Bug 1: Inner join → Left join (preserves all tx rows)")
print("Bug 2: No validate= → validate='one_to_one' (catches duplicates)")
print("Bug 3: Merge before aggregating → Aggregate before merging (correct order)")
print("\nAll bugs fixed.")